# Modelo baseline: Regresión lineal para la tasa de incidencia delictiva

En esta libreta se construye un modelo de  regresión lineal como baseline para analizar la relación entre variables socieconímicas y la tasa de incidencia delictiva. 

El objetivo es obtener una primera referencia del poder explicativo de los datos, identificar variables relevantes y establecer una base sobre la cual construir modelos más complejos. 

## Setup

In [1]:
import os

In [3]:
import numpy as np
import pandas as pd
import dotenv

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [5]:
import mlflow

In [6]:
from incidencia_delictiva.config import PROCESSED_DATA_DIR as processed_dir

### Tracking

Para registrar los modelos utilizaremos mlflow.

Cargamos las variables de entorno: 

In [7]:
dotenv.load_dotenv()

True

Configuramos el entorno para trackear los experimentos en el repositorio de dagshub: 

In [12]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment('baseline_delitos')

2026/04/29 11:49:01 INFO mlflow.tracking.fluent: Experiment with name 'baseline_delitos' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ba45d29b43a44abc8fc6918e66ec0bc6', creation_time=1777488541266, experiment_id='1', last_update_time=1777488541266, lifecycle_stage='active', name='baseline_delitos', tags={}>

In [15]:
os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD")

## Data

Utilizamos el dataset baseline\*. Este conjunto se compone de variables socieconómicas y posee definida la variable objetivo `tasa_delitos`.


\* La construcción de este fue documentado en la libreta [`dataset_modelado.ipynb`](dataset_modelado.ipynb) y el diccionario de las variables se encuentra en [`docs/data/baseline_dataset.md`](../docs/data/dataset_baseline.md). 

Definimos la ruta al archivo: 

In [17]:
dataset_baseline_path = processed_dir / "dataset_baseline.parquet"
assert dataset_baseline_path.exists() and dataset_baseline_path.is_file()

Cargamos el dataset: 

In [18]:
df = pd.read_parquet(dataset_baseline_path)

In [19]:
df.head()

,año,cvegeo,total_delitos,poblacion_total,indice_marginacion_normalizado_2020,porcentaje_analfabetismo,porcentaje_sin_agua_entubada,porcentaje_viviendas_hacinamiento,fn_pobreza_porcentaje,fn_pobreza_extrema_porcentaje,...,fn_carencia_seguridad_social_porcentaje,fn_poblacion,poblacion_urbano,nomgeo,area_km2,region,zona_metropolitana,es_frontera,tasa_delitos,prop_urbano
0,2015,03002,1284.0,64022,0.900254,4.153327,2.928495,21.225063,24.1,3.2,...,44.0,69902.0,45643.0,MulegÃ©,30681.679871,noroeste,0,0,2005.560589,0.652957
1,2015,03009,323.0,18052,0.921029,2.212159,2.857143,19.351351,34.1,2.9,...,43.8,19285.0,15085.0,Loreto,4586.260765,noroeste,0,0,1789.275427,0.782214
2,2015,04005,25.0,31917,0.871965,8.412576,2.288114,33.156372,66.7,16.6,...,63.7,37545.0,26529.0,HecelchakÃ¡n,1265.514509,sureste,0,1,78.328164,0.706592
3,2015,04008,47.0,11452,0.864073,8.777618,0.743332,35.261436,53.9,10.6,...,63.6,12092.0,9415.0,Tenabo,1050.953181,sureste,0,1,410.408662,0.778614
4,2015,05021,53.0,6539,0.904328,2.313781,0.382380,15.103903,16.8,0.7,...,31.4,6523.0,4315.0,Nadadores,711.737579,norte,0,1,810.521486,0.661505


In [20]:
df.shape

(26038, 21)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26038 entries, 0 to 26037
Data columns (total 21 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   año                                      26038 non-null  int64  
 1   cvegeo                                   26038 non-null  object 
 2   total_delitos                            26038 non-null  float64
 3   poblacion_total                          26038 non-null  int64  
 4   indice_marginacion_normalizado_2020      26038 non-null  float64
 5   porcentaje_analfabetismo                 26038 non-null  float64
 6   porcentaje_sin_agua_entubada             26038 non-null  float64
 7   porcentaje_viviendas_hacinamiento        26038 non-null  float64
 8   fn_pobreza_porcentaje                    26017 non-null  float64
 9   fn_pobreza_extrema_porcentaje            26017 non-null  float64
 10  fn_vulnerable_ingresos_porcentaje        26017

Los valores nulos: 

In [22]:
df.isnull().sum()

año                                           0
cvegeo                                        0
total_delitos                                 0
poblacion_total                               0
indice_marginacion_normalizado_2020           0
porcentaje_analfabetismo                      0
porcentaje_sin_agua_entubada                  0
porcentaje_viviendas_hacinamiento             0
fn_pobreza_porcentaje                        21
fn_pobreza_extrema_porcentaje                21
fn_vulnerable_ingresos_porcentaje            21
fn_carencia_seguridad_social_porcentaje      21
fn_poblacion                                 21
poblacion_urbano                           8084
nomgeo                                        0
area_km2                                      0
region                                        0
zona_metropolitana                            0
es_frontera                                   0
tasa_delitos                                  0
prop_urbano                             

- Las columnas `fn_*` poseen valores nulos, pero la cantidad es insignificante. 

- Las columnas `poblacion_urbano`, `prop_urbano` poseen aproximadamente el 30\% de valores nulos. 

Del diccionario de datos, observamos que existen variables redundantes y otras que no son de utilidad para el modelado. 

## Features

Definimos la variable objetivo: 

In [23]:
target = 'tasa_delitos'

Las variables `cvegeo`, `nomgeo` son indicadores de municipio, por lo que no aportan información reelevante al modelo. 

Las variables redundantes: 

- `total_delitos` (tasa ya la usa directamente)
- `poblacion_total` (influye en tasa)
- `fn_poblacion` (redundante)
- `poblacion_urbano` (deriva a proporcion urbano)

\* De momento, no analizaremos correlación de las variables. 

Definimos las variables a utilizazar para el modelado: 

In [24]:
exclude_cols = [
    'cvegeo', 'nomgeo', 
    'tasa_delitos',
    'total_delitos',
    'poblacion_total',
    'fn_poblacion', 
    'poblacion_urbano'
]

Las features: 

In [25]:
features = [col for col in df.columns if col not in exclude_cols]

La matriz de diseño: 

In [26]:
X = df[features]

La variable respuesta: 

In [27]:
y = df[target]

### Null values

La matriz de diseño posee valores nulos. Por simplicidad, consideraremos el uso de la mediana. 

In [28]:
cols_wnull = [
    'fn_pobreza_porcentaje', 
    'fn_pobreza_extrema_porcentaje', 
    'fn_vulnerable_ingresos_porcentaje', 
    'fn_carencia_seguridad_social_porcentaje',
    'prop_urbano'
]

X[cols_wnull] = (
    X[cols_wnull]
    .fillna(
        X[cols_wnull].median()
    )
)

/tmp/ipykernel_8796/1100572308.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[cols_wnull] = (


In [29]:
X.isnull().sum()

año                                        0
indice_marginacion_normalizado_2020        0
porcentaje_analfabetismo                   0
porcentaje_sin_agua_entubada               0
porcentaje_viviendas_hacinamiento          0
fn_pobreza_porcentaje                      0
fn_pobreza_extrema_porcentaje              0
fn_vulnerable_ingresos_porcentaje          0
fn_carencia_seguridad_social_porcentaje    0
area_km2                                   0
region                                     0
zona_metropolitana                         0
es_frontera                                0
prop_urbano                                0
dtype: int64

### Categorical

La feature `region` es de tipo categorica. Utilizamos One-Hot Encoding.

In [30]:
dummies_region = pd.get_dummies(X['region'], dtype=int)
X = pd.concat([X, dummies_region], axis=1)
X = X.drop('region', axis=1)

In [31]:
'region' not in X.columns

True

In [32]:
X.columns

Index(['año', 'indice_marginacion_normalizado_2020',
       'porcentaje_analfabetismo', 'porcentaje_sin_agua_entubada',
       'porcentaje_viviendas_hacinamiento', 'fn_pobreza_porcentaje',
       'fn_pobreza_extrema_porcentaje', 'fn_vulnerable_ingresos_porcentaje',
       'fn_carencia_seguridad_social_porcentaje', 'area_km2',
       'zona_metropolitana', 'es_frontera', 'prop_urbano', 'centro-norte',
       'centro-sur', 'noreste', 'noroeste', 'norte', 'occidente', 'sur',
       'sureste'],
      dtype='object')

Guardamos los nombres de las features para futuras referencias: 

In [40]:
feature_names = pd.DataFrame(X.columns, columns=['feature_name'])

## Train/Test split

Consideraremos un train/test split básico con proporcion 80-20 y una random seed de 42 para asegurar reproducibililidad. 

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Conviene mencionar que las variables socieconómicas corresponden a los resultados del censo de 2020, mientras que la variable objetivo (`tasa_delictiva`) captura la tasa de incidencia delictiva en el periodo 2015-2025. Si bien esto puede introducir sesgos en la interpretación, en esta etapa lo consideramos como una simplificación razonable, ya que el objetivo es construir un modelo exploratorio.

## OLS

En este modelo consideraremos un escalado estándar de los datos. Definimos la pipeline:

In [34]:
linreg = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

El entrenamiento: 

In [35]:
_ = linreg.fit(X_train, y_train)

Las predicciones: 

In [36]:
y_pred = linreg.predict(X_test)

Para la evaluación del modelo, considereamos las métricsa RMSE y $R^2$. 

In [37]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'RMSE: {rmse:.4f}')
print(f'R2: {r2:.4f}')

RMSE: 522.1317
R2: 0.4125


El RMSE es de 522, lo que indica que el modelo presenta un desempeño moderado. 

El coeficiente de determinación ($R^2$) indica que el modelo logra explicar aproximadamente el 41\% de la variabilidad en la tasa de incidencia delictiva. 

Si bien no es un ajuste alto, es rezonable para una primera aproximación con variables socieconómicas. 

La importancia de las variables la medimos usando los coeficientes asociados al modelo: 

In [39]:
coeffs = linreg.named_steps['model'].coef_

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coef': coeffs
}).sort_values(by='coef', key=abs, ascending=False)

feature_importance.head(10)

,feature,coef
5,fn_pobreza_porcentaje,-359.030348
6,fn_pobreza_extrema_porcentaje,244.238722
1,indice_marginacion_normalizado_2020,241.210797
14,centro-sur,141.491695
16,noroeste,-91.808148
3,porcentaje_sin_agua_entubada,86.926628
0,año,85.503258
9,area_km2,78.327641
17,norte,-51.205219
18,occidente,-48.479400


Los coeficientes del modelo muestran algunas relaciones relevantes: 

- Variables asociadas a pobreza (fn_pobreza_porcentaje, fn_pobreza_extrema_porcentaje) aparecen entre las más influyentes, aunque con signos opuestos, lo que sugiere posibles efectos diferenciados o problemas de colinealidad.
- El índice de marginación muestra una relación positiva con la tasa delictiva.
- Variables de infraestructura, como elm porcentaje de viviendas sin agua entubada, presentan relación positiva.
- La variable año tiene un coeficiente positivo, lo que podría indicar una tendencia temporal ascendente en la incidencia delictiva.
- Las variables geográficas (`centro-sur`, `noroeste`, `norte`) capturan diferencias importantes.

Con lo anterior, trackeamos el modelo usando mlflow: 

In [44]:
with mlflow.start_run(run_name='linear_regression_baseline'):

    # metrics
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('r2', r2)

    # params
    mlflow.log_param('model', 'linear_regression')
    mlflow.log_param('n_features', X.shape[1])

    # features
    feature_names.to_csv("feature_names.csv")
    mlflow.log_artifact("feature_names.csv")

    # model
    mlflow.sklearn.log_model(linreg, 'model')

    # coeffs as artifacts
    feature_importance.to_csv("feature_importance_linear.csv", index=False)
    mlflow.log_artifact("feature_importance_linear.csv")

2026/04/29 12:04:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/29 12:04:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run linear_regression_baseline at: https://dagshub.com/SebastianIsaacRV/proyecto_AAA_2026-1_equipo4.mlflow/#/experiments/1/runs/d179aee875c5494293883f23030600ce
🧪 View experiment at: https://dagshub.com/SebastianIsaacRV/proyecto_AAA_2026-1_equipo4.mlflow/#/experiments/1


## Ridge

Para contrastar con el modelo baseline (OLS), consideramos el modelo Ridge que utiliza la penalización $l_2$. El objetivo es evaluar un modelo que obtenga coeficientes más robustos y con ello evaluar la multicolinealidad de las features. 

In [45]:
ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)

rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2_ridge = r2_score(y_test, y_pred_ridge)

print(f"Ridge RMSE: {rmse_ridge:.4f}")
print(f"Ridge R2: {r2_ridge:.4f}")

Ridge RMSE: 522.1310
Ridge R2: 0.4125


El modelo Ridge presenta prácticamente el mismo desempeño que la regresión lineal simple: 

- RMSE: 522.13
- $R^22$: 0.41

Esto es esperable dado que la regularización aplicada ($\alpha$ = 1.0) no genera cambios significativos en el ajuste del modelo. Conviene mencionar, que se realizaron experimentos con multiples valores de $\alpha$ y el resultado permaneció igual. 

In [51]:
ridge_coeffs = ridge.named_steps["model"].coef_

ridge_importance_ridge = pd.DataFrame({
    "feature": X.columns,
    "coef": ridge_coeffs
}).sort_values(by="coef", key=abs, ascending=False)

ridge_importance.head(10)

,feature,coef
5,fn_pobreza_porcentaje,-358.876364
6,fn_pobreza_extrema_porcentaje,244.084073
1,indice_marginacion_normalizado_2020,241.128526
14,centro-sur,141.474427
16,noroeste,-91.786403
3,porcentaje_sin_agua_entubada,86.906676
0,año,85.498049
9,area_km2,78.318037
17,norte,-51.185785
18,occidente,-48.467761


Los coeficientes se mantienen muy similares a los del modelo lineal. 

La regularización no modificó ni la magnitud de los coeficientes ni el orden de las variables, lo que sugiere que, aunque existe colinealidad, no afecta la estabilidad del modelo baseline. 

Trackeamos el experimento: 

In [52]:
with mlflow.start_run(run_name='ridge_baseline'):

    # metrics
    mlflow.log_metric('rmse', rmse_ridge)
    mlflow.log_metric('r2', r2_ridge)

    # params
    mlflow.log_param('model', 'ridge')
    mlflow.log_param('n_features', X.shape[1])

    # features
    feature_names.to_csv("feature_names.csv")
    mlflow.log_artifact("feature_names.csv")

    # model
    mlflow.sklearn.log_model(ridge, 'model')

    # coeffs as artifacts
    ridge_importance.to_csv("feature_importance_ridge.csv", index=False)
    mlflow.log_artifact("feature_importance_ridge.csv")

2026/04/29 12:14:22 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/29 12:14:22 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run ridge_baseline at: https://dagshub.com/SebastianIsaacRV/proyecto_AAA_2026-1_equipo4.mlflow/#/experiments/1/runs/8746deb01b924aaca11501fd7f11d277
🧪 View experiment at: https://dagshub.com/SebastianIsaacRV/proyecto_AAA_2026-1_equipo4.mlflow/#/experiments/1


## Conclusión

Los resultados en ambos modelos (OLS y Ridge) son prácticamente identicos. Esto sugiere que

- Las variables predictoras incluidas explican la variable respuesta.
- Factores como pobreza, marginación e infraestructura aparecen como variables importantes.
- La regularización no aporta mejoras significatiavas en esta etapa.

Sin embargo, el modelo presenta limitaciones importantes: 

- Desajuste temporal (variables explicativas de 2020 y variable objetivo 2015-2020)
- Multicolinealidad entre variables predictoras (pobreza y marginacion son altamrnte relacionadas a la marginación)

Estos resultados deben de interpretarse como base exploratoria. 